In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# ─── Data (pivoted from separate_models_comparison.csv) ─────────
# Columns: Market | Trend | LSTM | RF | XGBoost | MLP | CNN | Transformer | SVM | Samples
col_headers = ['Market', 'Trend', 'LSTM', 'RF', 'XGBoost', 'MLP', 'CNN', 'Transformer', 'SVM', 'Samples']
col_widths  = [0.07, 0.09, 0.09, 0.09, 0.10, 0.09, 0.09, 0.13, 0.09, 0.10]

# (market, trend, [LSTM, RF, XGB, MLP, CNN, TR, SVM], samples)
rows = [
    ['BTC',  'Downtrend', 56.67, 55.00, 55.00, 50.00, 45.00, 43.33, 43.33, 460],
    ['BTC',  'Uptrend',   46.09, 64.06, 49.22, 63.28, 63.28, 36.72, 63.28, 776],
    ['Gold', 'Downtrend', 40.33, 52.49, 57.46, 37.57, 32.04, 70.17, 40.33, 1148],
    ['Gold', 'Uptrend',   64.85, 51.98, 53.47, 64.36, 67.33, 64.36, 67.33, 1117],
    ['Thai', 'Downtrend', 66.00, 66.00, 66.00, 66.00, 66.00, 66.00, 61.00, 647],
    ['Thai', 'Uptrend',   52.50, 56.25, 50.62, 67.50, 65.00, 38.12, 51.25, 969],
    ['UK',   'Downtrend', 47.99, 52.35, 56.38, 58.05, 58.39, 44.30, 60.07, 1474],
    ['UK',   'Uptrend',   66.47, 31.79, 34.10, 72.25, 49.13, 69.36, 63.01, 1019],
    ['US',   'Downtrend', 50.17, 50.84, 50.84, 54.21, 59.93, 45.79, 51.18, 1466],
    ['US',   'Uptrend',   80.71, 77.86, 42.14, 39.29, 19.29, 35.71, 26.43, 1270],
]

# ─── Layout ──────────────────────────────────────────────────────
ROW_H    = 0.60
HEADER_H = 0.72
FIG_W    = 18.0
N_ROWS   = len(rows)
FIG_H    = HEADER_H + N_ROWS * ROW_H + 0.20

BLACK = '#000000'
WHITE = '#ffffff'
GRAY  = '#f2f2f2'
LGRAY = '#cccccc'

market_bg = {
    'BTC':  GRAY,
    'Gold': WHITE,
    'Thai': GRAY,
    'UK':   WHITE,
    'US':   GRAY,
}

matplotlib.rcParams['font.family'] = ['DejaVu Sans']
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
ax.set_xlim(0, 1)
ax.set_ylim(0, FIG_H)
ax.axis('off')
fig.patch.set_facecolor(WHITE)

def cx(i):
    return sum(col_widths[:i])

# ── Header ────────────────────────────────────────────────────────
hdr_top = FIG_H - 0.10
hdr_bot = hdr_top - HEADER_H

ax.add_patch(mpatches.FancyBboxPatch(
    (0, hdr_bot), 1, HEADER_H,
    boxstyle='square,pad=0', facecolor=BLACK, edgecolor=BLACK,
    linewidth=2, clip_on=False))

for ci, (label, w) in enumerate(zip(col_headers, col_widths)):
    if ci > 0:
        ax.plot([cx(ci), cx(ci)], [hdr_bot, hdr_top],
                color=WHITE, linewidth=0.8, zorder=5)
    ax.text(cx(ci) + w / 2, (hdr_top + hdr_bot) / 2, label,
            ha='center', va='center',
            fontsize=11, fontweight='bold', color=WHITE)

ax.plot([0, 1], [hdr_bot, hdr_bot], color=BLACK, linewidth=2)

# ── Rows ──────────────────────────────────────────────────────────
prev_market = None
for ri, row_data in enumerate(rows):
    ry0 = hdr_bot - ri * ROW_H
    ry1 = ry0 - ROW_H
    market = row_data[0]
    trend  = row_data[1]
    accs   = row_data[2:9]   # 7 model accuracies
    samp   = row_data[9]
    best_acc = max(accs)

    ax.add_patch(mpatches.FancyBboxPatch(
        (0, ry1), 1, ROW_H,
        boxstyle='square,pad=0', facecolor=market_bg[market],
        edgecolor='none', clip_on=False))

    is_new = (market != prev_market)
    ax.plot([0, 1], [ry0, ry0],
            color=BLACK if is_new else LGRAY,
            linewidth=1.5 if is_new else 0.5)

    for ci, w in enumerate(col_widths):
        if ci > 0:
            ax.plot([cx(ci), cx(ci)], [ry1, ry0],
                    color=LGRAY, linewidth=0.6, zorder=5)

        if ci == 0:
            val = market
            fw, fs, fst = 'bold', 11, 'normal'
        elif ci == 1:
            val = trend
            fw  = 'bold' if trend == 'Uptrend' else 'normal'
            fs  = 11
            fst = 'normal' if trend == 'Uptrend' else 'italic'
        elif 2 <= ci <= 8:
            acc = accs[ci - 2]
            val = f'{acc:.2f}'
            is_best = (abs(acc - best_acc) < 1e-6)
            fw  = 'bold' if is_best else 'normal'
            fs  = 11 if is_best else 10.5
            fst = 'normal'
        else:    # Samples
            val = f'{samp:,}'
            fw, fs, fst = 'normal', 10.5, 'normal'

        ax.text(cx(ci) + w / 2, (ry0 + ry1) / 2, val,
                ha='center', va='center',
                fontsize=fs, fontweight=fw, fontstyle=fst, color=BLACK)

    prev_market = market

# ── Border ───────────────────────────────────────────────────────
bot = hdr_bot - N_ROWS * ROW_H
ax.plot([0, 1], [bot, bot], color=BLACK, linewidth=2)
ax.plot([0, 0], [bot, hdr_top], color=BLACK, linewidth=2)
ax.plot([1, 1], [bot, hdr_top], color=BLACK, linewidth=2)

# ── Save ─────────────────────────────────────────────────────────
out_dir  = '/Users/oattao/project/p-e/ipynb/image'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'table9_model_accuracy.png')

plt.tight_layout(pad=0.1)
plt.savefig(out_path, dpi=200, bbox_inches='tight',
            facecolor=WHITE, pad_inches=0.12)
plt.show()
print(f'Saved → {out_path}')